# BRSR Data Preprocessing Pipeline — CSV Edition

**Input:** `brsr-data/csv/` — 1200+ raw CSV files (one per company)

**Output:** `brsr-data/clean/` — one `*_clean.csv` per company + `quality_report.csv`

**Checks performed (no data is removed):**
- Schema verification (expected 12 columns)
- Null/missing detection on critical fields: `symbol`, `element`, `value`
- Corrupted/unreadable file detection
- Period field consistency across each filing

In [1]:
import pandas as pd
import os
from tqdm import tqdm

In [2]:
# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
INPUT_DIR  = os.path.join("brsr-data", "csv")    # raw CSVs from scraper
OUTPUT_DIR = os.path.join("brsr-data", "clean")  # clean CSVs go here

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Expected column schema (lowercase for comparison only)
EXPECTED_COLUMNS = {
    "symbol", "companyname", "fyfrom", "fyto",
    "submissiondate", "element", "value",
    "contextref", "period", "dimensions", "unit", "decimals"
}

# Fields that must not be null
CRITICAL_FIELDS = ["symbol", "element", "value"]

print(f"Input  : {INPUT_DIR}")
print(f"Output : {OUTPUT_DIR}")

Input  : brsr-data/csv
Output : brsr-data/clean


In [3]:
# ---------------------------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------------------------

def check_schema(df):
    """Returns (ok, missing_cols, extra_cols). Uses lowercase comparison."""
    actual  = {c.strip().lower() for c in df.columns}
    missing = sorted(EXPECTED_COLUMNS - actual)
    extra   = sorted(actual - EXPECTED_COLUMNS)
    return (len(missing) == 0), missing, extra


def null_summary(df):
    """Returns {field: null_count} for each critical field."""
    col_map = {c.strip().lower(): c for c in df.columns}
    return {
        field: (
            int(df[col_map[field]].isnull().sum())
            if field in col_map else "column_missing"
        )
        for field in CRITICAL_FIELDS
    }


def period_check(df):
    """
    Returns (consistent: bool, unique_periods: list).
    Ignores blank / ' to ' placeholder values.
    Consistent = only one distinct period across the file.
    """
    col_map = {c.strip().lower(): c for c in df.columns}
    if "period" not in col_map:
        return True, []
    raw = df[col_map["period"]].dropna().astype(str).str.strip()
    unique = sorted({p for p in raw if p not in ("", "to", " to ")})
    return (len(unique) <= 1), unique

In [4]:
# ---------------------------------------------------------------------------
# MAIN LOOP
# ---------------------------------------------------------------------------
all_files = sorted(f for f in os.listdir(INPUT_DIR) if f.endswith(".csv"))
print(f"Found {len(all_files)} CSV files\n")

quality_log = []

for file in tqdm(all_files, desc="Processing"):
    filepath = os.path.join(INPUT_DIR, file)
    out_path = os.path.join(OUTPUT_DIR, file.replace(".csv", "_clean.csv"))

    log = dict(
        file=file, status="", rows=None,
        schema_ok=None, missing_columns="", extra_columns="",
        null_symbol=None, null_element=None, null_value=None,
        period_consistent=None, unique_periods="",
        output_file="", error=""
    )

    # 1. READ ---------------------------------------------------------------
    df = None
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            df = pd.read_csv(filepath, dtype=str, encoding=enc, on_bad_lines="warn")
            break
        except UnicodeDecodeError:
            continue
        except Exception as e:
            log["status"] = "FAILED_UNREADABLE"
            log["error"]  = str(e)
            break

    if df is None:
        if not log["status"]:
            log["status"] = "FAILED_CORRUPT"
            log["error"]  = "Could not decode with utf-8 / latin-1 / cp1252"
        quality_log.append(log)
        continue

    log["rows"] = len(df)

    # 2. SCHEMA -------------------------------------------------------------
    schema_ok, missing, extra = check_schema(df)
    log["schema_ok"]       = schema_ok
    log["missing_columns"] = ", ".join(missing)
    log["extra_columns"]   = ", ".join(extra)

    # 3. NULLS --------------------------------------------------------------
    nulls = null_summary(df)
    log["null_symbol"]  = nulls["symbol"]
    log["null_element"] = nulls["element"]
    log["null_value"]   = nulls["value"]
    has_nulls = any(isinstance(v, int) and v > 0 for v in nulls.values())

    # 4. PERIOD -------------------------------------------------------------
    period_ok, unique_periods = period_check(df)
    log["period_consistent"] = period_ok
    log["unique_periods"]    = " | ".join(unique_periods)

    # 5. STATUS -------------------------------------------------------------
    if not schema_ok:
        log["status"] = "SCHEMA_ERROR"
    elif has_nulls or not period_ok:
        log["status"] = "WARNING"
    else:
        log["status"] = "OK"

    # 6. WRITE CLEAN CSV (all data preserved) -------------------------------
    df.to_csv(out_path, index=False, encoding="utf-8")
    log["output_file"] = os.path.basename(out_path)

    quality_log.append(log)

print("\nProcessing complete.")

Found 1227 CSV files



Processing: 100%|██████████| 1227/1227 [00:14<00:00, 83.48it/s]


Processing complete.


In [5]:
# ---------------------------------------------------------------------------
# SAVE QUALITY REPORT
# ---------------------------------------------------------------------------
report_df   = pd.DataFrame(quality_log)
report_path = os.path.join(OUTPUT_DIR, "quality_report.csv")
report_df.to_csv(report_path, index=False)

print("=" * 50)
print("         DATA QUALITY REPORT")
print("=" * 50)
print(f"Total files        : {len(all_files)}")
print(f"OK                 : {(report_df['status'] == 'OK').sum()}")
print(f"WARNING            : {(report_df['status'] == 'WARNING').sum()}")
print(f"SCHEMA_ERROR       : {(report_df['status'] == 'SCHEMA_ERROR').sum()}")
print(f"FAILED             : {report_df['status'].str.startswith('FAILED').sum()}")
print(f"Period issues      : {(report_df['period_consistent'] == False).sum()}")
print(f"\nReport saved to    : {report_path}")
print(f"Clean CSVs in      : {OUTPUT_DIR}/")

         DATA QUALITY REPORT
Total files        : 1227
OK                 : 0
SCHEMA_ERROR       : 0
FAILED             : 0
Period issues      : 1227

Report saved to    : brsr-data/clean/quality_report.csv
Clean CSVs in      : brsr-data/clean/


In [6]:
# ---------------------------------------------------------------------------
# INSPECT PROBLEM FILES (optional)
# ---------------------------------------------------------------------------
problems = report_df[report_df["status"] != "OK"]
if not problems.empty:
    print(f"{len(problems)} files with issues:\n")
    display(
        problems[["file", "status", "missing_columns",
                  "null_element", "null_value",
                  "period_consistent", "error"]].reset_index(drop=True)
    )
else:
    print("All files passed quality checks!")

1227 files with issues:



,file,status,missing_columns,null_element,null_value,period_consistent,error
0,360ONE_2024_2025.csv,WARNING,,0,41,False,
1,3IINFOLTD_2024_2025.csv,WARNING,,0,23,False,
2,3MINDIA_2024_2025.csv,WARNING,,0,11,False,
3,5PAISA_2024_2025.csv,WARNING,,0,90,False,
4,63MOONS_2024_2025.csv,WARNING,,0,5,False,
...,...,...,...,...,...,...,...
1222,ZENTEC_2024_2025.csv,WARNING,,0,30,False,
1223,ZFCVINDIA_2024_2025.csv,WARNING,,0,9,False,
1224,ZOTA_2024_2025.csv,WARNING,,0,2,False,
1225,ZYDUSLIFE_2024_2025.csv,WARNING,,0,2,False,
